# 🎭 Facial Emotion Recognition (Deep Convolutional Neural Network - CNN)

This project implements an end-to-end **Facial Emotion Classifier** using a Deep Convolutional Neural Network (CNN) in TensorFlow/Keras. The model classifies 48x48 grayscale facial images from the FER2013 dataset into 7 fundamental emotion categories: **Angry, Disgust, Fear, Happy, Neutral, Sad, and Surprise**.

### 🧠 Algorithm & Architecture Overview:
* **Deep CNN Architecture:** 3 stacked Convolutional blocks with increasing filter dimensions (64 $\rightarrow$ 128 $\rightarrow$ 256). Each block contains dual `Conv2D` layers, `BatchNormalization`, `MaxPooling2D`, and spatial `Dropout` (0.25) to prevent overfitting.
* **Data Augmentation:** Real-time `ImageDataGenerator` pipeline applying random rotations, translations, shearing, zooming, and horizontal flips.
* **Optimization & Callbacks:** Compiled with the `Adam` optimizer and `Categorical Crossentropy` loss, monitored via `ModelCheckpoint`, `EarlyStopping`, and `ReduceLROnPlateau` for dynamic learning rate adjustments.

### Step 1: Import Libraries & Environment Setup

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

### Step 2: Define Hyperparameters & Model Configurations

In [ ]:
IMG_SIZE = 48
BATCH_SIZE = 64
EPOCHS = 30
MODEL_PATH = "emotion_model.h5"

### Step 3: Set Dataset Directory Paths

In [ ]:
TRAIN_DIR = "Dataset/train"
TEST_DIR = "Dataset/test"

### Step 4: Configure Data Augmentation & Preprocessing Pipelines

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

### Step 5: Load Training & Validation Data Generators

In [ ]:
print("[INFO] Loading dataset from directories...")
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

### Step 6: Initialize Sequential Model Architecture

In [ ]:
print("[INFO] Compiling CNN model...")

model = Sequential()

### Step 7: Convolutional Block 1 (64 Filters & Max Pooling)

In [ ]:
model.add(Conv2D(64, (3, 3), padding='same', activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

### Step 8: Convolutional Block 2 (128 Filters & Max Pooling)

In [ ]:
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

### Step 9: Convolutional Block 3 (256 Filters & Max Pooling)

In [ ]:
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

### Step 10: Fully Connected Dense Classifier & Regularization

In [ ]:
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

### Step 11: Output Layer (7-Class Softmax)

In [ ]:
model.add(Dense(7, activation='softmax'))

### Step 12: Model Compilation (Adam Optimizer & Crossentropy Loss)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

### Step 13: Configure Callbacks (Checkpoint, EarlyStopping, ReduceLROnPlateau)

In [ ]:
callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
]

### Step 14: Model Training & Validation Loop

In [ ]:
print(f"\n[INFO] Starting training for {EPOCHS} epochs...")
model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

### Step 15: Training Completion Verification

In [ ]:
print(f"\n[SUCCESS] Model training complete. Saved to '{MODEL_PATH}'!")